# ANA-071: frozen inputs and numerical reproduction
This notebook checks the completed RUN-184 archive, recorded objective and native
oracle predictions. It performs no optimization and changes no fitted parameters.
Passing these checks establishes computational consistency, not TrpB adequacy.

In [1]:
from pathlib import Path
import hashlib, json, sys, platform, importlib.metadata
import numpy as np
from scipy.special import expit
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'labproject.yaml').exists())
ANA = ROOT / 'analyses/2026-11-24_ANA-071_run-184-trpb-architecture-calibration-and-landscape-adequacy'
BUNDLE = ROOT / 'data/processed/aggregate-thermodynamic-gnk-calibration/RUN-184'
OUT = BUNDLE / 'outputs'
TABLES = ANA / 'tables'; TABLES.mkdir(exist_ok=True)
def read(name): return json.loads((OUT / name).read_text())
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
result = read('result.json'); empirical = read('empirical_architecture.json')
trials = read('trials.json'); config = read('config.json')
checks = {}

## Archive identities and disjoint panels
Every collected file, frozen runtime payload and package snapshot is verified
against the archive's recorded SHA-256 identities. Both panels retain actual
assay replicates. Confirmation genotypes and generator seeds are disjoint from
calibration; this is not an independent biological dataset.

In [2]:
manifest = json.loads((BUNDLE / 'collection_manifest.json').read_text())
for name, entry in manifest['files'].items():
    p = BUNDLE / name
    assert p.stat().st_size == entry['bytes'] and sha(p) == entry['sha256'], name
frozen = json.loads((BUNDLE / 'frozen_runtime/prelaunch_manifest.json').read_text())
for name, digest in frozen['files'].items():
    assert sha(BUNDLE / 'frozen_runtime' / name) == digest, name
provenance = read('provenance.json')
for name, digest in provenance['source_sha256'].items():
    assert sha(OUT / 'source_snapshot' / name) == digest, name
cal = np.load(OUT / 'inputs.npz', allow_pickle=False)
con = np.load(OUT / 'confirmation_inputs.npz', allow_pickle=False)
pred = np.load(OUT / 'confirmation_predictions.npz', allow_pickle=False)
assert len(cal['sequences']) == len(con['sequences']) == 8192
assert not (set(map(tuple, cal['sequences'])) & set(map(tuple, con['sequences'])))
assert len(np.unique(cal['sequences'], axis=0)) == len(np.unique(con['sequences'], axis=0)) == 8192
assert np.array_equal(con['sequences'], pred['sequences'])
assert pred['values'].shape == (4,8192,2)
assert np.array_equal(pred['values'][:,:,0], pred['values'][:,:,1])
assert np.array_equal(pred['seeds'], result['confirmation_seeds'])
assert not set(result['calibration_seeds']) & set(result['confirmation_seeds'])
assert np.isclose(cal['fitness'].mean(), result['phenotype_origin'])
assert np.isclose(cal['fitness'].std(), result['phenotype_scale'])
checks['archive'] = {'collection_files': len(manifest['files']),
    'frozen_payloads': len(frozen['files']), 'source_snapshot_files': len(provenance['source_sha256']),
    'disjoint_genotype_panels': True, 'fresh_confirmation_seeds': True}
print(checks['archive'])

{'collection_files': 224, 'frozen_payloads': 152, 'source_snapshot_files': 18, 'disjoint_genotype_panels': True, 'fresh_confirmation_seeds': True}


## Objective, trial states and convergence
The distance averages squared residuals in 18 summaries after division by
fixed bootstrap-derived scales. It is neither prediction error nor a likelihood.
Failed attempts remain failed; they are not assigned invented objective values.

In [3]:
complete = [t for t in trials if t['state']=='COMPLETE']
failed = [t for t in trials if t['state']=='FAIL']
assert (len(trials),len(complete),len(failed)) == (100,37,63)
assert all(t['value'] is None and 'failure' in t['diagnostics'] for t in failed)
best = min(complete,key=lambda t:t['value'])
assert best['number'] == result['best_trial'] == 63
scales = np.asarray(empirical['fixed_scales'])
recomputed_scales = np.sqrt(np.var(empirical['bootstrap_summaries'],axis=0,ddof=1)+config['scale_floor']**2)
np.testing.assert_allclose(scales,recomputed_scales,rtol=1e-13)
cal_loss = np.mean(((np.mean(best['diagnostics']['summaries_by_seed'],axis=0)-empirical['summary'])/scales)**2)
conf = result['confirmation_architecture']
con_loss = np.mean(((np.mean(conf['simulated'],axis=0)-conf['target'])/scales)**2)
np.testing.assert_allclose([cal_loss,con_loss],[result['calibration_loss'],result['confirmation_loss']],rtol=1e-13)
fits = [empirical['inference'],*empirical['bootstrap_inference'],conf['empirical_inference'],*conf['synthetic_inference']]
for trial in complete: fits.extend(trial['diagnostics']['auxiliary_inference'])
assert all(f['converged'] and f['convergence']['selected_cv_converged'] for f in fits)
for fitted in fits:
    selected=[fold for fold in fitted['cv_records'] if fold['penalty']==fitted['selected_penalty']]
    assert len(selected)==config['inference']['n_folds']
    assert len({fold['fold'] for fold in selected})==config['inference']['n_folds']
    for fit in [fitted['convergence'],*selected]:
        assert fit['optimizer_success'] and fit['informative']
        assert fit['projected_gradient_per_observation'] <= config['inference']['gradient_tolerance']
checks['objective'] = {'attempts':100,'complete':37,'failed':63,'calibration_loss':cal_loss,
    'confirmation_loss':con_loss,'checked_selected_auxiliary_fits':len(fits),
    'fixed_scales_recomputed':True,'selected_full_fit_and_cv_convergence':True}
print(checks['objective'])

{'attempts': 100, 'complete': 37, 'failed': 63, 'calibration_loss': np.float64(24.034136531319845), 'confirmation_loss': np.float64(27.34629870164042), 'checked_selected_auxiliary_fits': 174, 'fixed_scales_recomputed': True, 'selected_full_fit_and_cv_convergence': True}


## Replay the native oracle from the frozen source
The imported code must resolve inside the archived snapshot, independently of
any editable package installation. All four stored predictions are replayed.

In [4]:
snapshot = OUT / 'source_snapshot'
sys.path.insert(0,str(snapshot))
import adaptenv
from adaptenv.calibration import build_calibrated_landscape
assert Path(adaptenv.__file__).resolve().is_relative_to(snapshot.resolve())
errors = []
for i,seed in enumerate(pred['seeds']):
    model = build_calibrated_landscape(result['best_parameters'],cal['additive_matrix'],
        seed=int(seed),target_degree=result['target_degree'])
    replay = model.evaluate_batch(con['sequences'])
    error = float(np.max(np.abs(replay-pred['values'][i,:,0])))
    assert error < 1e-12
    errors.append(error)
checks['oracle_replay'] = {'maximum_absolute_error_by_seed':errors,'frozen_source_import':True}
print(checks['oracle_replay'])

{'maximum_absolute_error_by_seed': [5.204170427930421e-18, 5.637851296924623e-18, 6.071532165918825e-18, 5.204170427930421e-18], 'frozen_source_import': True}


## Reconstruct the empirical auxiliary target and held-out predictions
This empirical-coefficient model is a separate object from the synthetic
generator. Evaluating its frozen calibration coefficients on confirmation data
tests prediction by those fitted coefficients; it does not score the generator
by genotype identity, which the generator was not designed to reproduce.

In [5]:
c = np.load(OUT/'empirical_coefficients.npz',allow_pickle=False)
def auxiliary_predict(x):
    z = c['intercept'] + c['additive'][np.arange(4)[None,:],x].sum(axis=1)
    for k,(i,j) in enumerate(c['pairs']): z += c['pairwise'][k,x[:,i],x[:,j]]
    standardized = c['lower']+(c['upper']-c['lower'])*expit(z)
    return standardized,expit(z)
def coefficient_shape(x):
    x=np.abs(x).ravel(); squared=x*x
    rms=np.sqrt(np.mean(squared))
    n=max(1,int(np.ceil(.1*len(x))))
    return np.quantile(x/rms,[.5,.9,.99]),np.sort(squared)[-n:].sum()/squared.sum()
v1=np.sum(c['additive']**2)/20;v2=np.sum(c['pairwise']**2)/400
q1,t1=coefficient_shape(c['additive']);q2,t2=coefficient_shape(c['pairwise'])
_,occupancy=auxiliary_predict(cal['sequences'])
standardized=(cal['fitness']-result['phenotype_origin'])/result['phenotype_scale']
summaries=np.array([np.log1p(v1),np.log1p(v2),v2/(v1+v2),*q1,*q2,t1,t2,
    *np.quantile(standardized,[.1,.5,.9]),float(c['lower']),float(c['upper']),
    np.mean(occupancy<.05),np.mean(occupancy>.95)])
np.testing.assert_allclose(summaries,empirical['summary'],rtol=1e-11,atol=1e-12)
standard_pred,_=auxiliary_predict(con['sequences'])
native_pred=result['phenotype_origin']+result['phenotype_scale']*standard_pred
y=con['fitness'];ym=y.mean(axis=1)
r2_replicates=1-np.sum((y-native_pred[:,None])**2)/np.sum((y-y.mean())**2)
r2_means=1-np.sum((ym-native_pred)**2)/np.sum((ym-ym.mean())**2)
checks['empirical_auxiliary']={'18_architecture_summaries_reconstructed':True,
    'confirmation_r2_individual_replicates':float(r2_replicates),
    'confirmation_r2_genotype_means':float(r2_means)}
environment={'python':platform.python_version(),'platform':platform.platform(),
    'packages':{n:importlib.metadata.version(n) for n in ['numpy','scipy','pandas','matplotlib','nbformat','nbclient','ipykernel']}}
(TABLES/'validation_checks.json').write_text(json.dumps({'status':'passed','checks':checks},indent=2)+'\n')
(TABLES/'analysis_environment.json').write_text(json.dumps(environment,indent=2)+'\n')
print(checks['empirical_auxiliary'])
print('Computational validation passed. Scientific adequacy is assessed in the figure notebooks.')

{'18_architecture_summaries_reconstructed': True, 'confirmation_r2_individual_replicates': 0.720063981156215, 'confirmation_r2_genotype_means': 0.7353885138485423}
Computational validation passed. Scientific adequacy is assessed in the figure notebooks.
